In [18]:
import pandas as pd
import random
from collections import defaultdict
import json

In [22]:
# Parameters
ITEMS_PER_QUESTION = 5
QUESTIONS_PER_CLUSTER = 2

# Load data
reps = pd.read_csv('../data/results/spectral_13_reps.csv')
anime_old = pd.read_csv('../data/anime_old.csv')

# Manual English title mappings
manual_english_titles = {
    48707: "The Way of the Househusband Part 2",
    40852: "Dr. Stone: Stone Wars",
    5028: "Major Season 5",
    14397: "Chihayafuru Season 2",
    18: "Initial D Fourth Stage",
    137: "Hunter x Hunter OVA",
    39063: "Fairy Gone",
    35756: "Comic Girls",
    10507: "Inazuma Eleven Go",
    35459: "My Hero Academia: Training of the Dead",
    12565: "Fate/Prototype",
    18661: "Kamisama Kiss OVA",
    40615: "The Stranger by the Shore",
    12729: "High School DxD OVA",
    30885: "Noragami Aragoto OVA",
    31706: "Fate/kaleid liner Prisma Illya 3rei",
    36286: "Re:Zero Memory Snow",
    9062: "Angel Beats! Specials",
    40056: "Deca-Dence"
}

# Merge with English titles
data = reps.merge(
    anime_old[['MAL_ID', 'English name', 'Genres']], 
    left_on='anime_id', right_on='MAL_ID', how='left'
)

# Build cluster groups once
cluster_groups = defaultdict(list)
for _, row in data.iterrows():
    anime_id = row['anime_id']
    if anime_id in manual_english_titles:
        display_title = manual_english_titles[anime_id]
    elif pd.notna(row['English name']) and row['English name'] not in ['Unknown', '']:
        display_title = row['English name'].strip()
    else:
        display_title = row['title']
    
    cluster_groups[row['cluster_id']].append({
        'anime_id': anime_id, 
        'title': display_title, 
        'genres': row['Genres'] if pd.notna(row['Genres']) else 'Unknown'
    })


def generate_questions(seed):
    random.seed(seed)
    cluster_reserved = {}
    cluster_extras = {}
    questions = []

    # Reserve anime and extras
    for cluster_id, anime_list in cluster_groups.items():
        needed_for_own = (ITEMS_PER_QUESTION - 1) * QUESTIONS_PER_CLUSTER
        if len(anime_list) >= needed_for_own:
            shuffled = anime_list[:]
            random.shuffle(shuffled)
            cluster_reserved[cluster_id] = shuffled[:needed_for_own]
            cluster_extras[cluster_id] = shuffled[needed_for_own:]

    # Build questions
    for cluster_id in sorted(cluster_reserved.keys()):
        reserved_anime = cluster_reserved[cluster_id][:]
        for _ in range(QUESTIONS_PER_CLUSTER):
            same_cluster = reserved_anime[:ITEMS_PER_QUESTION - 1]
            reserved_anime = reserved_anime[ITEMS_PER_QUESTION - 1:]

            odd_one = None
            other_cluster_ids = [cid for cid in cluster_extras.keys() if cid != cluster_id]
            random.shuffle(other_cluster_ids)
            for other_cluster_id in other_cluster_ids:
                if cluster_extras[other_cluster_id]:
                    odd_one = cluster_extras[other_cluster_id].pop(0)
                    break
            if odd_one is None:
                break

            options = same_cluster + [odd_one]
            random.shuffle(options)
            correct_idx = next(i for i, a in enumerate(options) if a['anime_id'] == odd_one['anime_id'])
            questions.append({
                "question": "Which anime is the odd one out?",
                "options": [a['title'] for a in options],
                "correct": options[correct_idx]['title']
            })
    return questions


# === Generate 3 versions ===
seeds = [42, 99, 123]
versions = {}
for i, seed in enumerate(seeds, start=1):
    versions[f"Version {i}"] = generate_questions(seed)

# === Print Apps Script–ready JS ===
print("const versions = " + json.dumps(versions, indent=2) + ";")

const versions = {
  "Version 1": [
    {
      "question": "Which anime is the odd one out?",
      "options": [
        "Junjo Romantica 2",
        "Just Because!",
        "To LOVE Ru Darkness 2",
        "The Troubled Life of Miss Kotoura",
        "Infinite Stratos"
      ],
      "correct": "Junjo Romantica 2"
    },
    {
      "question": "Which anime is the odd one out?",
      "options": [
        "Bakuman.",
        "Waiting in the Summer",
        "Boarding School Juliet",
        "My Little Monster",
        "The Pet Girl of Sakurasou"
      ],
      "correct": "Bakuman."
    },
    {
      "question": "Which anime is the odd one out?",
      "options": [
        "Tokyo Godfathers",
        "SHIMONETA:A Boring World Where the Concept of Dirty Jokes Doesn't Exist",
        "Kino's Journey",
        "Berserk:The Golden Age Arc II - The Battle for Doldrey",
        "Summer Wars"
      ],
      "correct": "SHIMONETA:A Boring World Where the Concept of Dirty Jokes Doesn't Exis